# 設備保全ドキュメントでつくる RAG ハンズオン（章0・章1）

経産省・厚労省の公開資料（スマート保安事例集、リスクアセスメントマニュアル、労働災害事例）と Snowflake のガイドを使い、
**PDF解析 → チャンク化 → Cortex Search → 回答生成** までを順番に体験します。

| 章 | 内容 |
|---|---|
| 0 | 環境準備（データベース、ステージ、GitHub からのデータ取得） |
| 1-1 | AI_PARSE_DOCUMENT で PDF を解析 |
| 1-2 | SPLIT_TEXT_RECURSIVE_CHARACTER でチャンク化 |
| 1-3 | Cortex Search サービスの作成（GUI または SQL） |
| 1-4 | 検索してみる |
| 1-5 | 検索結果を使って AI_COMPLETE で回答を生成（RAG） |

> 上のセルから順番に実行してください。

## 章0 環境準備

### 0-1 データベース・ウェアハウス・ステージを作成する

- `RAG_HANDSON_DB.DOCS` スキーマに、すべてのオブジェクトを作ります。
- ステージは **サーバーサイド暗号化（SNOWFLAKE_SSE）** と **ディレクトリテーブル** を有効にします。AI_PARSE_DOCUMENT はこの設定のステージを前提にしています。

In [ ]:
%%sql -r setup_objects
USE ROLE ACCOUNTADMIN;

CREATE DATABASE IF NOT EXISTS RAG_HANDSON_DB;
CREATE SCHEMA IF NOT EXISTS RAG_HANDSON_DB.DOCS;

-- AI_PARSE_DOCUMENT は MEDIUM 以下を推奨（大きくしても速くならない）
CREATE WAREHOUSE IF NOT EXISTS RAG_WH
  WAREHOUSE_SIZE = 'XSMALL'
  AUTO_SUSPEND = 60
  AUTO_RESUME = TRUE;

USE WAREHOUSE RAG_WH;
USE SCHEMA RAG_HANDSON_DB.DOCS;

CREATE STAGE IF NOT EXISTS RAW_DOCS
  DIRECTORY = (ENABLE = TRUE)
  ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

### 0-2 GitHub リポジトリを Snowflake に接続する

ハンズオンのデータは GitHub で公開しています。Snowflake の **Git リポジトリ統合** を使うと、ファイルをダウンロードしてアップロードし直さなくても、SQL だけでステージにコピーできます。

> **講師メモ**：`sfc-gh-kichikawa` と `rag-handson-cortex-search` は公開リポジトリが決まったら差し替える。

In [ ]:
%%sql -r setup_git
CREATE OR REPLACE API INTEGRATION RAG_GIT_API_INTEGRATION
  API_PROVIDER = git_https_api
  API_ALLOWED_PREFIXES = ('https://github.com/sfc-gh-kichikawa/')
  ENABLED = TRUE;

CREATE OR REPLACE GIT REPOSITORY RAG_HANDSON_REPO
  API_INTEGRATION = RAG_GIT_API_INTEGRATION
  ORIGIN = 'https://github.com/sfc-gh-kichikawa/rag-handson-cortex-search.git';

ALTER GIT REPOSITORY RAG_HANDSON_REPO FETCH;

### 0-3 資料をステージにコピーする

Git リポジトリの `data/` フォルダにある PDF と HTML を、0-1 で作ったステージ `RAW_DOCS` にコピーします。フォルダ構成（`A_smart_hoan/` など）はそのまま保たれます。

In [ ]:
%%sql -r copy_files
COPY FILES
  INTO @RAW_DOCS
  FROM @RAG_HANDSON_REPO/branches/main/data/
  PATTERN = '.*[.](pdf|html)';

ALTER STAGE RAW_DOCS REFRESH;

> **CoCo で fix する体験（ハンズオン）**
>
> 次のセルはワークスペースのファイルをステージにコピーする SQL ですが、
> **あえてエラーが出るようになっています。** まずそのまま実行してください。
>
> **エラーの原因:** `COPY FILES ... FROM 'snow://workspace/...'` のワークスペースパスは
> 環境ごとに異なります（ユーザー名やワークスペース名が含まれるため）。
> コード上のパスが実際の環境と一致しない場合、
> `Workspace '...' does not exist or not authorized` というエラーになります。
>
> **対処方法:** エラーが表示されたら、セル右上の **fix ボタン** をクリックしてください。
> CoCo がワークスペースパスを自動で修正してくれます。

In [ ]:
-- ワークスペースのファイルを RAW_DOCS ステージにコピー
COPY FILES
  INTO @RAG_HANDSON_DB.DOCS.RAW_DOCS
  FROM 'snow://workspace/USER$.PUBLIC."20261013_RAG"/versions/live/data/'
  PATTERN = '.*[.](pdf|html)';

ALTER STAGE RAG_HANDSON_DB.DOCS.RAW_DOCS REFRESH;

In [ ]:
%%sql -r list_files
SELECT
    RELATIVE_PATH,
    SPLIT_PART(RELATIVE_PATH, '/', 1) AS FOLDER,
    ROUND(SIZE / 1024) AS SIZE_KB
FROM DIRECTORY(@RAW_DOCS)
ORDER BY RELATIVE_PATH;

## 章1 テキストRAG

### 1-1 AI_PARSE_DOCUMENT で PDF を解析する

- `mode: 'LAYOUT'`：見出し・表・段落の構造を保ったまま Markdown に変換します。
- `page_split: true`：ページごとに結果を分けます。回答の引用元にページ番号を出せるようになります。
- 画像の抽出（`extract_images`）は章2で扱います。

約90ページの解析に数分かかります。待っている間に、次のセルの説明を読んでおいてください。

In [ ]:
%%sql -r parse_docs
CREATE OR REPLACE TABLE PARSED_DOCS AS
SELECT
    RELATIVE_PATH,
    SPLIT_PART(RELATIVE_PATH, '/', 1) AS FOLDER,
    AI_PARSE_DOCUMENT(
        TO_FILE('@RAW_DOCS', RELATIVE_PATH),
        {'mode': 'LAYOUT', 'page_split': true}
    ) AS PARSED
FROM DIRECTORY(@RAW_DOCS)
WHERE RELATIVE_PATH ILIKE '%.pdf' OR RELATIVE_PATH ILIKE '%.html';

SELECT FOLDER, COUNT(*) AS FILES, SUM(PARSED:metadata:pageCount::INT) AS PAGES
FROM PARSED_DOCS
GROUP BY FOLDER
ORDER BY FOLDER;

解析結果をページ単位の行に展開します。

In [ ]:
%%sql -r pages
CREATE OR REPLACE TABLE DOC_PAGES AS
SELECT
    d.RELATIVE_PATH,
    d.FOLDER,
    p.value:index::INT + 1 AS PAGE_NO,
    p.value:content::VARCHAR AS CONTENT
FROM PARSED_DOCS d,
     LATERAL FLATTEN(input => d.PARSED:pages) p;

-- 解析結果の確認：味噌チョッパーの事例（101464）
SELECT RELATIVE_PATH, PAGE_NO, LEFT(CONTENT, 600) AS CONTENT_PREVIEW
FROM DOC_PAGES
WHERE RELATIVE_PATH ILIKE '%101464%'
ORDER BY PAGE_NO;

> **確認ポイント**：日本語が崩れずに読み取れているか、表が Markdown の表になっているかを見てください。
> AI_PARSE_DOCUMENT の公式の対応言語一覧に日本語は含まれていないため、精度は資料によって差が出ます。

### 1-2 チャンク化する

検索の精度を上げるため、ページの本文を短い単位（チャンク）に分けます。

- `SPLIT_TEXT_RECURSIVE_CHARACTER(テキスト, 'markdown', チャンクの長さ, 重なり)`
- `'markdown'` を指定すると、見出しや段落の区切りを優先して分割します。
- 長さは文字数で指定します。Cortex Search は 512 トークン以下のチャンクを推奨しているため、日本語では 600 文字程度を目安にします。

In [ ]:
%%sql -r chunks
CREATE OR REPLACE TABLE DOC_CHUNKS AS
SELECT
    p.RELATIVE_PATH,
    p.FOLDER,
    p.PAGE_NO,
    c.index AS CHUNK_NO,
    c.value::VARCHAR AS CHUNK,
    CASE p.FOLDER
        WHEN 'A_smart_hoan'      THEN 'スマート保安事例集'
        WHEN 'B_risk_assessment' THEN 'リスクアセスメントマニュアル'
        WHEN 'C_cases'           THEN '労働災害事例'
        WHEN 'D_snowflake_guide' THEN 'Snowflakeガイド'
    END AS DOC_TYPE
FROM DOC_PAGES p,
     LATERAL FLATTEN(input => SNOWFLAKE.CORTEX.SPLIT_TEXT_RECURSIVE_CHARACTER(p.CONTENT, 'markdown', 600, 100)) c;

ALTER TABLE DOC_CHUNKS SET CHANGE_TRACKING = TRUE;

SELECT DOC_TYPE, COUNT(*) AS CHUNKS, ROUND(AVG(LENGTH(CHUNK))) AS AVG_CHARS
FROM DOC_CHUNKS
GROUP BY DOC_TYPE
ORDER BY DOC_TYPE;

### 1-3 Cortex Search サービスを作成する

**方法A：GUI で作成する（推奨）**

1. 左メニュー **AI & ML » 検索** を開き、右上の**作成**ボタンを押す
2. ロール `ACCOUNTADMIN`、ウェアハウス `RAG_WH`、データベース `RAG_HANDSON_DB`、スキーマ `DOCS` を選ぶ
3. サービス名に `RAG検索` と入力
4. データの選択で **テーブルまたはビュー** を選び、`DOC_CHUNKS` を選ぶ
5. 検索列の選択では `CHUNK` を選ぶ
6. 属性列を選択では `DOC_TYPE` を選ぶ
7. サービスに含む列を選択では `RELATIVE_PATH`, `PAGE_NO` を選ぶ
8. 検索サービスを構成ではターゲットラグは `1 day` を選択
9. 埋め込みモデルに **snowflake-arctic-embed-l-v2.0**（多言語対応）を選び、あとはデフォルトのまま作成

> GUI で作るとサービス名が二重引用符付き（`"RAG_SEARCH"`）になる場合があります。以降のセルで見つからない場合は、方法B で作り直してください。

**方法B：SQL で作成する**

GUI を使わない場合、または作り直す場合は次のセルを実行します。

In [ ]:
%%sql -r create_search
CREATE OR REPLACE CORTEX SEARCH SERVICE RAG_SEARCH
  ON CHUNK
  ATTRIBUTES DOC_TYPE
  WAREHOUSE = RAG_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT CHUNK, RELATIVE_PATH, PAGE_NO, DOC_TYPE
    FROM DOC_CHUNKS
);

### 1-4 検索してみる

`SEARCH_PREVIEW` で、自然言語の質問に近いチャンクを取り出します。キーワードが一致しなくても、意味が近いチャンクが返ることを確認してください。

1. 「運転を止めずに手を入れて起きた事故は？」→ C のはさまれ事例がヒット
2. 「ボイラーの安全弁の点検で注意することは？」→ B の保護方策がヒット
3. 「タンクの中の点検で注意する有害物は？」→ C の硫化水素・一酸化炭素事例がヒット
4. 「ドローンで高所点検している事例は？」→ A の関西電力の事例がヒット
5. 「スモールスタートで設備保全のデジタル化を進めるには？」→ A の成功要因 + D の Snowflake ガイドが横断でヒット
6. 「チョッパーの巻き込まれ事故を防ぐ方策は？」→ C の事例101464 + B のガード・エネルギー遮断がヒット

In [ ]:
%%sql
SELECT
    r.value:DOC_TYPE::VARCHAR      AS DOC_TYPE,
    r.value:RELATIVE_PATH::VARCHAR AS SOURCE,
    r.value:PAGE_NO::INT           AS PAGE_NO,
    LEFT(r.value:CHUNK::VARCHAR, 200) AS CHUNK
FROM TABLE(FLATTEN(PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'RAG_HANDSON_DB.DOCS."RAG検索"',
        '{"query": "運転を止めずに手を入れて起きた事故", "columns": ["CHUNK","RELATIVE_PATH","PAGE_NO","DOC_TYPE"], "limit": 5}'
    )
):results)) r;

フィルタを使うと、資料の種類（`DOC_TYPE`）で絞り込めます。ここでは労働災害事例だけを対象にします。

In [ ]:
%%sql
SELECT
    r.value:DOC_TYPE::VARCHAR      AS DOC_TYPE,
    r.value:RELATIVE_PATH::VARCHAR AS SOURCE,
    LEFT(r.value:CHUNK::VARCHAR, 200) AS CHUNK
FROM TABLE(FLATTEN(PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'RAG_HANDSON_DB.DOCS."RAG検索"',
        '{"query": "タンクの中の点検で注意すること", "columns": ["CHUNK","RELATIVE_PATH","DOC_TYPE"], "filter": {"@eq": {"DOC_TYPE": "労働災害事例"}}, "limit": 5}'
    )
):results)) r;

### 1-5 検索結果を使って回答を生成する（RAG）

1. Cortex Search で質問に関係するチャンクを取り出す
2. 取り出したチャンクと質問をまとめてプロンプトにする
3. AI_COMPLETE で回答を生成する

下のセルの `question` を書き換えて、何度か試してみてください。

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.core import Root

session = get_active_session()
search = (Root(session)
          .databases["RAG_HANDSON_DB"]
          .schemas["DOCS"]
          .cortex_search_services['"RAG検索"'])

MODEL = "claude-sonnet-4-5"   # リージョンで使えない場合は 'llama4-maverick' などに変更

def ask(question, limit=5, doc_type=None):
    kwargs = {"query": question,
              "columns": ["CHUNK", "RELATIVE_PATH", "PAGE_NO", "DOC_TYPE"],
              "limit": limit}
    if doc_type:
        kwargs["filter"] = {"@eq": {"DOC_TYPE": doc_type}}
    hits = search.search(**kwargs).results

    context = "\n\n".join(
        f"[資料{i+1}] {h['DOC_TYPE']} / {h['RELATIVE_PATH']} p.{h['PAGE_NO']}\n{h['CHUNK']}"
        for i, h in enumerate(hits))
    prompt = (
        "あなたは設備保全と労働安全の専門家です。以下の資料だけを根拠に、日本語で簡潔に回答してください。\n"
        "回答の最後に、根拠にした資料番号を [資料1] のように示してください。資料に答えがない場合は、わからないと答えてください。\n\n"
        f"# 資料\n{context}\n\n# 質問\n{question}")

    answer = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, prompt]).collect()[0][0]
    print(answer)
    print("\n--- 参照した資料 ---")
    for i, h in enumerate(hits):
        print(f"[資料{i+1}] {h['DOC_TYPE']} / {h['RELATIVE_PATH']} p.{h['PAGE_NO']}")

question = "チョッパーの巻き込まれ事故を防ぐには、どのような保護方策がありますか？"
ask(question)

In [ ]:
# 資料を横断する質問の例
ask("設備保全のデジタル化をスモールスタートで進めるには、どうすればよいですか？")

### 1-6 CoWork で検索できるようにする（Cortex Agent）

章1-5 では Notebook の中で検索と回答を試しました。
ここからは、**Snowflake CoWork**（ブラウザのチャット画面）から同じ検索ができるようにします。

仕組みは次のとおりです。

1. **Cortex Agent** を作成し、検索ツールとして Cortex Search サービスを登録する
2. Agent は CoWork に自動的に表示される
3. ユーザーがチャット欄に質問を入力すると、Agent が Cortex Search で資料を検索し、LLM が回答を生成する

つまり、章1-5 の `ask()` 関数でやっていたことを、Agent がコード不要で行ってくれます。

In [ ]:
%%sql
CREATE OR REPLACE AGENT RAG_HANDSON_DB.DOCS.RAG_AGENT
  FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    あなたは設備保全と労働安全の専門家です。
    日本語で簡潔に回答してください。
    回答の根拠となった資料名とページ番号を必ず示してください。
    資料に答えがない場合は「資料に該当する情報がありません」と答えてください。
  sample_questions:
    - question: "運転を止めずに手を入れて起きた事故を教えてください"
    - question: "ボイラーの安全弁の点検で注意することは？"
    - question: "設備保全のデジタル化をスモールスタートで進めるには？"
    - question: "チョッパーの巻き込まれ事故を防ぐ方策は？"
tools:
  - tool_spec:
      type: cortex_search
      name: "設備保全ドキュメント検索"
      description: "スマート保安事例集、リスクアセスメントマニュアル、労働災害事例、Snowflakeガイドを横断検索します"
tool_resources:
  設備保全ドキュメント検索:
    search_service: RAG_HANDSON_DB.DOCS."RAG検索"
    max_results: 5
    stage_path: "@RAG_HANDSON_DB.DOCS.RAW_DOCS"
    relative_path_column: "RELATIVE_PATH"
$$;

-- CoWork からアクセスするために USAGE を付与
GRANT USAGE ON DATABASE RAG_HANDSON_DB TO ROLE PUBLIC;
GRANT USAGE ON SCHEMA RAG_HANDSON_DB.DOCS TO ROLE PUBLIC;
GRANT USAGE ON AGENT RAG_HANDSON_DB.DOCS.RAG_AGENT TO ROLE PUBLIC;
GRANT USAGE ON CORTEX SEARCH SERVICE RAG_HANDSON_DB.DOCS."RAG検索" TO ROLE PUBLIC;

-- CoWork に Agent を登録（登録済みの場合はスキップ）
EXECUTE IMMEDIATE $$
BEGIN
  ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT
    ADD AGENT RAG_HANDSON_DB.DOCS.RAG_AGENT;
EXCEPTION
  WHEN OTHER THEN NULL;
END;
$$;

## 章1のまとめ

| ステップ | 使った機能 |
|---|---|
| PDF の解析 | AI_PARSE_DOCUMENT（LAYOUT, page_split） |
| チャンク化 | SPLIT_TEXT_RECURSIVE_CHARACTER |
| 検索 | Cortex Search（ハイブリッド検索＋フィルタ） |
| 回答生成 | AI_COMPLETE |

次の章2では、PDF の中の図やイラストも検索できるようにします（マルチモーダルRAG）。

## 章2 マルチモーダルRAG：画像も検索対象にする

章1では PDF のテキストだけを検索していました。
章2では、PDF に含まれる **図やイラスト** も検索対象にします。

仕組みは次のとおりです。

1. `AI_PARSE_DOCUMENT` に `extract_images: true` を付けて、PDF から画像を抽出する
2. 抽出した画像を Vision 対応の LLM に渡して、日本語の説明文（キャプション）を生成する
3. 説明文をチャンクに追加して、Cortex Search を再構築する
4. 「感電のイラスト」「ドローン点検の写真」のように、**図の内容で検索** できることを確認する

### 2-1 画像を抽出する

In [ ]:
%%sql
CREATE OR REPLACE TABLE RAG_HANDSON_DB.DOCS.PARSED_WITH_IMAGES AS
SELECT
    RELATIVE_PATH,
    SPLIT_PART(RELATIVE_PATH, '/', 1) AS FOLDER,
    AI_PARSE_DOCUMENT(
        TO_FILE('@RAG_HANDSON_DB.DOCS.RAW_DOCS', RELATIVE_PATH),
        {'mode': 'LAYOUT', 'extract_images': true}
    ) AS PARSED
FROM DIRECTORY(@RAG_HANDSON_DB.DOCS.RAW_DOCS)
WHERE RELATIVE_PATH ILIKE '%.pdf';

-- 抽出された画像の数を確認
SELECT
    FOLDER,
    COUNT(*) AS FILES,
    SUM(ARRAY_SIZE(COALESCE(PARSED:images, ARRAY_CONSTRUCT()))) AS IMAGES
FROM RAG_HANDSON_DB.DOCS.PARSED_WITH_IMAGES
GROUP BY FOLDER
ORDER BY FOLDER;

### 2-2 アイコンや装飾を除外する

PDF にはヘッダーのロゴやページ番号のアイコンなど、検索に不要な小さい画像が含まれます。
バウンディングボックス（画像の位置と大きさ）を使って、幅または高さが小さい画像を除外します。

In [ ]:
%%sql
CREATE OR REPLACE TABLE RAG_HANDSON_DB.DOCS.DOC_IMAGES AS
SELECT
    d.RELATIVE_PATH,
    d.FOLDER,
    img.index AS IMG_INDEX,
    img.value:id::VARCHAR AS IMG_ID,
    img.value:image_base64::VARCHAR AS IMAGE_BASE64,
    (img.value:bottom_right_x::FLOAT - img.value:top_left_x::FLOAT) AS WIDTH,
    (img.value:bottom_right_y::FLOAT - img.value:top_left_y::FLOAT) AS HEIGHT
FROM RAG_HANDSON_DB.DOCS.PARSED_WITH_IMAGES d,
     LATERAL FLATTEN(input => d.PARSED:images) img
WHERE (img.value:bottom_right_x::FLOAT - img.value:top_left_x::FLOAT) >= 100
  AND (img.value:bottom_right_y::FLOAT - img.value:top_left_y::FLOAT) >= 100;

SELECT FOLDER, COUNT(*) AS IMAGES,
       ROUND(AVG(WIDTH)) AS AVG_W, ROUND(AVG(HEIGHT)) AS AVG_H
FROM RAG_HANDSON_DB.DOCS.DOC_IMAGES
GROUP BY FOLDER
ORDER BY FOLDER;

### 2-3 画像の説明文を生成する

抽出した画像を Vision 対応の LLM に渡して、日本語の説明文を生成します。
Base64 の画像データをデコードしてステージに保存し、`AI_COMPLETE` に渡します。

> 画像の枚数によっては数分かかります。

In [ ]:
from snowflake.snowpark.context import get_active_session
import json

session = get_active_session()

# 画像をステージに保存
session.sql("CREATE STAGE IF NOT EXISTS RAG_HANDSON_DB.DOCS.IMG_STAGE ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')").collect()

# 画像の Base64 データをステージにファイルとして書き出すプロシージャ
session.sql("""
CREATE OR REPLACE PROCEDURE RAG_HANDSON_DB.DOCS.SAVE_IMAGES_FROM_TABLE()
RETURNS VARCHAR
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python')
HANDLER = 'run'
AS
$$
import base64, os, tempfile, re

def run(session):
    rows = session.sql("SELECT RELATIVE_PATH, IMG_ID, IMAGE_BASE64 FROM RAG_HANDSON_DB.DOCS.DOC_IMAGES").collect()
    count = 0
    with tempfile.TemporaryDirectory() as td:
        for r in rows:
            b64 = r['IMAGE_BASE64']
            # Base64 のヘッダーを除去
            b64_data = re.sub(r'^data:image/[^;]+;base64,', '', b64)
            img_bytes = base64.b64decode(b64_data)
            # ファイル名: フォルダ_ファイル名_画像ID.jpg
            fname = r['RELATIVE_PATH'].replace('/', '_') + '_' + r['IMG_ID'] + '.jpg'
            fpath = os.path.join(td, fname)
            with open(fpath, 'wb') as f:
                f.write(img_bytes)
            session.file.put(fpath, '@RAG_HANDSON_DB.DOCS.IMG_STAGE', auto_compress=False, overwrite=True)
            count += 1
    return f'{count} images saved'
$$
""").collect()

result = session.sql("CALL RAG_HANDSON_DB.DOCS.SAVE_IMAGES_FROM_TABLE()").collect()
print(result[0][0])

In [ ]:
%%sql
-- ステージのディレクトリを更新
ALTER STAGE RAG_HANDSON_DB.DOCS.IMG_STAGE REFRESH;

-- 画像の説明文を生成
CREATE OR REPLACE TABLE RAG_HANDSON_DB.DOCS.IMAGE_CAPTIONS AS
SELECT
    i.RELATIVE_PATH,
    i.FOLDER,
    i.IMG_ID,
    AI_COMPLETE(
        'claude-sonnet-4-5',
        'この画像の内容を日本語で100文字以内で説明してください。設備保全や労働安全の文脈で、何が描かれているかを具体的に記述してください。',
        TO_FILE('@RAG_HANDSON_DB.DOCS.IMG_STAGE', REPLACE(i.RELATIVE_PATH, '/', '_') || '_' || i.IMG_ID || '.jpg')
    ) AS CAPTION
FROM RAG_HANDSON_DB.DOCS.DOC_IMAGES i;

SELECT FOLDER, COUNT(*) AS IMAGES, LEFT(CAPTION, 80) AS CAPTION_SAMPLE
FROM RAG_HANDSON_DB.DOCS.IMAGE_CAPTIONS
GROUP BY FOLDER, CAPTION
ORDER BY FOLDER
LIMIT 10;

### 2-4 説明文をチャンクに追加して Cortex Search を再構築する

画像の説明文を新しいチャンクとしてテーブルに追加し、Cortex Search サービスを作り直します。

In [ ]:
%%sql
-- 既存のテキストチャンク＋画像説明文のチャンクを統合
CREATE OR REPLACE TABLE RAG_HANDSON_DB.DOCS.DOC_CHUNKS_V2 AS
-- テキストチャンク（章1と同じ）
SELECT CHUNK, RELATIVE_PATH, PAGE_NO, DOC_TYPE
FROM RAG_HANDSON_DB.DOCS.DOC_CHUNKS
UNION ALL
-- 画像の説明文チャンク
SELECT
    '【図の説明】' || CAPTION AS CHUNK,
    RELATIVE_PATH,
    0 AS PAGE_NO,
    CASE SPLIT_PART(RELATIVE_PATH, '/', 1)
        WHEN 'A_smart_hoan'      THEN 'スマート保安事例集'
        WHEN 'B_risk_assessment' THEN 'リスクアセスメントマニュアル'
        WHEN 'C_cases'           THEN '労働災害事例'
        WHEN 'D_snowflake_guide' THEN 'Snowflakeガイド'
    END AS DOC_TYPE
FROM RAG_HANDSON_DB.DOCS.IMAGE_CAPTIONS;

ALTER TABLE RAG_HANDSON_DB.DOCS.DOC_CHUNKS_V2 SET CHANGE_TRACKING = TRUE;

SELECT DOC_TYPE, COUNT(*) AS CHUNKS FROM RAG_HANDSON_DB.DOCS.DOC_CHUNKS_V2 GROUP BY DOC_TYPE ORDER BY DOC_TYPE;

In [ ]:
%%sql
-- Cortex Search サービスを再作成（画像の説明文チャンクを含む）
CREATE OR REPLACE CORTEX SEARCH SERVICE RAG_HANDSON_DB.DOCS."RAG検索"
  ON CHUNK
  ATTRIBUTES DOC_TYPE
  WAREHOUSE = RAG_WH
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
AS (
    SELECT CHUNK, RELATIVE_PATH, PAGE_NO, DOC_TYPE
    FROM RAG_HANDSON_DB.DOCS.DOC_CHUNKS_V2
);

### 2-5 画像の内容で検索してみる

図やイラストの説明文がチャンクに入ったので、画像の内容でも検索できるようになりました。

In [ ]:
%%sql -r search_images
SELECT
    r.value:DOC_TYPE::VARCHAR      AS DOC_TYPE,
    r.value:RELATIVE_PATH::VARCHAR AS SOURCE,
    LEFT(r.value:CHUNK::VARCHAR, 200) AS CHUNK
FROM TABLE(FLATTEN(PARSE_JSON(
    SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
        'RAG_HANDSON_DB.DOCS."RAG検索"',
        '{"query": "感電に関するイラスト", "columns": ["CHUNK","RELATIVE_PATH","DOC_TYPE"], "limit": 5}'
    )
):results)) r;

In [ ]:
# 画像の内容を含めた RAG
ask("ドローンを使った高所点検の事例について、図も含めて教えてください")

In [ ]:
%%sql
-- 画像キャプション対応版の Agent を再作成し CoWork に反映
CREATE OR REPLACE AGENT RAG_HANDSON_DB.DOCS.RAG_AGENT
  FROM SPECIFICATION $$
models:
  orchestration: auto
instructions:
  response: |
    あなたは設備保全と労働安全の専門家です。
    日本語で簡潔に回答してください。
    回答の根拠となった資料名とページ番号を必ず示してください。
    図の説明が含まれる場合は、どのような図があるかも言及してください。
    資料に答えがない場合は「資料に該当する情報がありません」と答えてください。
  sample_questions:
    - question: "運転を止めずに手を入れて起きた事故を教えてください"
    - question: "ボイラーの安全弁の点検で注意することは？"
    - question: "設備保全のデジタル化をスモールスタートで進めるには？"
    - question: "チョッパーの巻き込まれ事故を防ぐ方策は？"
    - question: "ドローンを使った高所点検の事例について教えてください"
tools:
  - tool_spec:
      type: cortex_search
      name: "設備保全ドキュメント検索"
      description: "スマート保安事例集、リスクアセスメントマニュアル、労働災害事例、Snowflakeガイドをテキストと画像説明を含めて横断検索します"
tool_resources:
  設備保全ドキュメント検索:
    search_service: RAG_HANDSON_DB.DOCS."RAG検索"
    max_results: 5
    stage_path: "@RAG_HANDSON_DB.DOCS.RAW_DOCS"
    relative_path_column: "RELATIVE_PATH"
$$;

-- CoWork からアクセスするために USAGE を付与
GRANT USAGE ON DATABASE RAG_HANDSON_DB TO ROLE PUBLIC;
GRANT USAGE ON SCHEMA RAG_HANDSON_DB.DOCS TO ROLE PUBLIC;
GRANT USAGE ON AGENT RAG_HANDSON_DB.DOCS.RAG_AGENT TO ROLE PUBLIC;
GRANT USAGE ON CORTEX SEARCH SERVICE RAG_HANDSON_DB.DOCS."RAG検索" TO ROLE PUBLIC;

-- CoWork に Agent を登録（登録済みの場合はスキップ）
EXECUTE IMMEDIATE $$
BEGIN
  ALTER SNOWFLAKE INTELLIGENCE SNOWFLAKE_INTELLIGENCE_OBJECT_DEFAULT
    ADD AGENT RAG_HANDSON_DB.DOCS.RAG_AGENT;
EXCEPTION
  WHEN OTHER THEN NULL;
END;
$$;

## 章2のまとめ

| ステップ | 使った機能 |
|---|---|
| 画像の抽出 | AI_PARSE_DOCUMENT（extract_images: true） |
| アイコン除外 | bounding_box のサイズでフィルタ |
| 説明文の生成 | AI_COMPLETE（Vision） |
| チャンクに統合 | UNION ALL |
| 検索 | Cortex Search（テキスト＋画像説明文のハイブリッド検索） |

これで、テキストだけでなく **図やイラストの内容でも検索できる** マルチモーダル RAG が完成しました。

> **参加者への問いかけ**：自社の点検報告書に写真や図面が含まれている場合、同じ手順で画像も検索対象にできます。
> どのような資料で使えそうか、考えてみてください。